In [1]:
import sys
from pathlib import Path

import yaml
import torch
import torchinfo

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from training.loop import create_model

CONFIG_DIR = REPO_ROOT / "configs" / "heterogeneous_dataset"
DEFAULT_GRID = (16, 64, 32)  # (D, H, W)

/Users/ntt/anaconda3/envs/torch/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
CONFIG_STEMS = [
  # FNO
  "fno_m8x32x16_h64_hetero",
  "fno_m4x16x8_h64_hetero",
  "fno_m4x16x8_h128_hetero",
  # UNet
  "unet_d3_hetero",
  "unet_d4_hetero",
  # Transolver
  "transolver_hetero",
  "transolver_h128_s64_hetero",
  # Modulated LOGLO
  "modulated_loglo_hetero",
  "vanilla_loglo_hetero",
]


def load_config(stem: str) -> dict:
    with open(CONFIG_DIR / f"{stem}.yml") as f:
        return yaml.safe_load(f)


def grid_shape(model_cfg: dict) -> tuple[int, int, int]:
    if "H" in model_cfg:
        return model_cfg["H"], model_cfg["W"], model_cfg["D"]
    return DEFAULT_GRID


def summary_input_size(model_type: str, model_cfg: dict) -> tuple | list:
    d, h, w = grid_shape(model_cfg)
    if model_type == "modulated_loglo":
        return [
            (1, model_cfg["in_dim"], d, h, w),
            (1, model_cfg["action_channels"], d, h, w),
        ]
    if model_type in ("fno", "unet3d", "transolver"):
        return (1, model_cfg["in_channels"], d, h, w)
    if model_type == "vanilla_loglo":
        return (1, model_cfg["in_dim"], d, h, w)
    raise ValueError(f"Unsupported model type: {model_type}")

In [3]:
for stem in CONFIG_STEMS:
    cfg = load_config(stem)
    model_cfg = cfg["model"]
    model_type = model_cfg["type"]
    model = create_model(model_cfg, model_type)
    input_size = summary_input_size(model_type, model_cfg)

    print("=" * 100)
    print(f"{stem}  (type={model_type}, variant={'hetero' if cfg['data'].get('heterogeneous') else 'homo'})")
    print("=" * 100)
    print(torchinfo.summary(model, input_size=input_size))
    print()

fno_m8x32x16_h64_hetero  (type=fno, variant=hetero)
Layer (type:depth-idx)                        Output Shape              Param #
FNOWrapper                                    [1, 4, 16, 64, 32]        --
├─FNO: 1-1                                    [1, 4, 16, 64, 32]        --
│    └─GridEmbeddingND: 2-1                   [1, 13, 16, 64, 32]       --
│    └─ChannelMLP: 2-2                        [1, 64, 16, 64, 32]       --
│    │    └─ModuleList: 3-1                   --                        10,048
│    └─FNOBlocks: 2-3                         [1, 64, 16, 64, 32]       28,336,800
│    │    └─ModuleList: 3-14                  --                        (recursive)
│    │    └─ModuleList: 3-15                  --                        (recursive)
│    │    └─ModuleList: 3-16                  --                        (recursive)
│    │    └─ModuleList: 3-17                  --                        (recursive)
│    └─FNOBlocks: 2-4                         [1, 64, 16, 64, 32]     